# **Make dimensionality reduced versions of modalities**
On current embeddings:
- HnE, scRNAseq, BulkRNAseq, WES

On versions of data prior to current preprocessing:
- scRNAseq, BulkRNAseq, spatial

Clinical data embeddings are a concatenation of raw and MCA data so no need for this modality.

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [31]:
%load_ext autoreload
%autoreload 2
from gbmhackathon.s3_loader import load_s3, write_s3
from gbmhackathon.models.dim_reduction import *
from gbmhackathon.training.patientwise import PatientLearningDataset, collate_patient_wise
import torch
from torch.utils.data import DataLoader
from copy import deepcopy
from tqdm import tqdm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### **On current embeddings**

**For `PCA`, we will keep `99.9%` of explained variance. MDS implementation of sklearn does not allow access to eigenvalues so not possible to define output size based on explained variance.
For `UMAP` and `MDS` we will use a fixed size. Arbitrarily we fixed this to `64`.**

In [3]:
explained_var = 0.999
n_components = 64
seed = 6262

In [4]:
pca = PCA(explained_variance_threshold=explained_var)
umap = UMAP(n_components=n_components, random_state=seed)
mds = MDS(n_components=n_components)

In [5]:
dr_dict = {'pca':pca, 'umap':umap, 'mds':mds}

### Convenient way to load data

In [6]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
# "spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
# "clinical":"2025-05-25_14-36_new_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset = PatientLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=1.0)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 582
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_patient_wise, generator=torch.Generator(device=dataset.device))

Using device : cpu
By keeping 100.00% of dropout augmented samples we went from:
194 dropout samples (62.99% dropout in dataset) -- to --> 194 dropout samples (62.99% dropout in dataset)
Dataset size: 308


In [8]:
batch_all = [dataset.__getitem__(idx) for idx in dataset.ind2patient if 'd' not in dataset.ind2patient[idx]]
batch_all = collate_patient_wise(batch_all)

In [9]:
ind2patient = {i:batch_all[0][i] for i in range(len(batch_all[0]))}

In [10]:
HNE = batch_all[2]['hne']
SC = batch_all[2]['scRNA']
BULK = batch_all[2]['bulk']
WES = batch_all[2]['wes']

In [11]:
modality_dict = {'hne':(HNE, name_emb_dict['hne'], "embeddings_HnE_OptimusH0"), 
                 'scRNA':(SC, name_emb_dict['scRNA'], "wes_emb_V1"), 
                 'bulk':(BULK, name_emb_dict['bulk'], "bulk_emb_V1"), 
                 'wes':(WES, name_emb_dict['wes'], "scRNA_emb_V1")}

In [12]:
S3_FOLDER = "s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1"

In [13]:
for mod in modality_dict.keys():
    # print("\n", mod)
    data, name, root = modality_dict[mod]
    data_emb = load_s3(f"{S3_FOLDER}/{name}")
    fixed_id_map = {}
    for idx, pid1 in ind2patient.items():
        for pid in data_emb["data"].keys():
            if pid1 in pid:
                fixed_id_map[idx] = pid
                break
    for dr in dr_dict.keys():
        # print("\n", dr.upper())
        data_emb_copy = deepcopy(data_emb)
        if dr != 'mds':
            # Fit reducer
            dr_dict[dr](data)
            
            # Apply transformations to each sample individually
            # print("Before", data_emb_copy["data"][fixed_id_map[1]].size())
            for pid in data_emb_copy['data'].keys():
                data_emb_copy['data'][pid] = dr_dict[dr].transform(data_emb_copy['data'][pid]).squeeze()
            # print("After", data_emb_copy["data"][fixed_id_map[1]].size())
        else:
            # Fit reducer
            # print("Before", data_emb_copy["data"][fixed_id_map[1]].size())
            _, data_emb_copy["data"] = dr_dict[dr](data, fixed_id_map)
            # print("After", data_emb_copy["data"][fixed_id_map[1]].size())
        # write_s3(obj=data_emb_copy, save_name=root + f"_{dr}",folder="embedding_V1")

84 dimensions hold 99.90% of variance.
Effective compression ratio: 5.47%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_embeddings_HnE_OptimusH0_pca.pkl
Effective compression ratio: 4.17%


/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_embeddings_HnE_OptimusH0_umap.pkl
Effective compression ratio: 4.17%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_embeddings_HnE_OptimusH0_mds.pkl
53 dimensions hold 99.90% of variance.
Effective compression ratio: 1.73%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_wes_emb_V1_pca.pkl
Effective compression ratio: 2.08%


/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_wes_emb_V1_umap.pkl
Effective compression ratio: 2.08%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_wes_emb_V1_mds.pkl
62 dimensions hold 99.90% of variance.
Effective compression ratio: 2.02%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_bulk_emb_V1_pca.pkl
Effective compression ratio: 2.08%


/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_bulk_emb_V1_umap.pkl
Effective compression ratio: 2.08%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_bulk_emb_V1_mds.pkl
107 dimensions hold 99.90% of variance.
Effective compression ratio: 5.98%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_scRNA_emb_V1_pca.pkl
Effective compression ratio: 3.58%


/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_scRNA_emb_V1_umap.pkl
Effective compression ratio: 3.58%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_15-11_scRNA_emb_V1_mds.pkl


### **Using raw data**
### **Bulk**

In [13]:
from gbmhackathon.data import MosaicDataset
from foundation.singlecell_bulk import remove_duplicate_var_indices, replace_nan_var_indices

In [14]:
data = MosaicDataset.load_tabular()["bulk_rna"]['TPM counts']
ensembl_ids = data.index.astype(str).tolist() # retrieve gene IDS

data = remove_duplicate_var_indices(data, 'bulk')
print(f"DEBUG: Data after removing duplicate indices shape: {data.shape}")
data = replace_nan_var_indices(data, 'bulk')
print(f"DEBUG: Data after replacing NaN indices shape: {data.shape}")
data = data.T
data.index = [idx[:idx.index("_mRNA")] for idx in data.index]


DEBUG: Entering remove_duplicate_var_indices with mode: bulk
DEBUG: Mode is bulk, extracted data.columns
DEBUG: Found 0 duplicate variable indices.
DEBUG: Shape of data after removing duplicates: (49856, 104)
Removed 0 variables with duplicate indices.
DEBUG: Data after removing duplicate indices shape: (49856, 104)

DEBUG: Entering replace_nan_var_indices with mode: bulk
DEBUG: Mode is bulk, filling NaN in data.index
DEBUG: data.index is unique after filling NaN.
DEBUG: Data after replacing NaN indices shape: (49856, 104)


In [15]:
data

EnsemblID,ENSG00000227232,ENSG00000278267,ENSG00000238009,ENSG00000268903,ENSG00000269981,ENSG00000239906,ENSG00000241860,ENSG00000222623,ENSG00000279928,ENSG00000279457,...,ENSG00000198886,ENSG00000210176,ENSG00000210184,ENSG00000210191,ENSG00000198786,ENSG00000198695,ENSG00000210194,ENSG00000198727,ENSG00000210195,ENSG00000210196
HK_G_001a,2.646295,13.143915,0.159919,1.973038,17.833763,0.0,0.433552,0.0,0.000000,3.412212,...,4487.957109,5880.854365,3979.116125,268.555489,3272.283604,2868.060722,77.720542,2371.679925,1886.882054,1463.355896
HK_G_002a,4.818720,55.589013,0.000000,0.000000,0.739447,0.0,0.305600,0.0,0.000000,3.006484,...,7999.161194,2106.116405,1961.213863,100.564786,5207.307910,2438.034089,27.391687,3401.090500,916.376449,395.299645
HK_G_003a,4.988052,36.336981,0.000000,2.082649,1.581892,0.0,0.178300,0.0,0.394085,8.522059,...,5587.188444,2334.184272,2014.042979,120.223764,4787.118367,1856.501995,22.788410,2641.010368,864.479810,445.953862
HK_G_004a,10.826797,10.082953,0.000000,16.649114,37.018164,0.0,0.362821,0.0,0.000000,7.034730,...,7974.761885,2162.915118,1847.743797,151.291628,5295.427247,2505.744972,46.371840,4133.875087,723.731935,524.313538
HK_G_005a,10.334242,82.126826,0.000000,3.434253,0.702292,0.0,0.844348,0.0,0.000000,19.416834,...,6863.541287,1757.480063,1855.907177,129.221685,5162.056894,2984.544714,28.905922,3125.662463,707.143967,686.345615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HK_G_111b,1.985879,3.034980,0.415416,1.366746,17.077108,0.0,0.614303,0.0,0.000000,1.181839,...,4841.436846,4676.420951,4514.096094,219.459005,3132.923835,1151.199859,25.423459,2777.882178,1780.797739,1394.573491
HK_G_112a,1.614058,32.067527,0.000000,0.000000,0.548439,0.0,0.103027,0.0,0.000000,6.132148,...,2495.382865,2519.193067,1763.481059,190.856631,1702.233537,1111.063466,2.257341,1309.392576,856.661083,716.938286
HK_G_113b,0.000000,22.379758,0.000000,0.000000,0.000000,0.0,0.036605,0.0,0.000000,1.089351,...,5988.503053,22775.221814,19825.913492,1100.935068,3878.313641,3289.246956,166.418119,3026.671874,7818.735162,7969.228243
HK_G_114a,0.685128,3.402972,0.093157,3.677915,8.147962,0.0,0.015306,0.0,0.000000,1.822064,...,3525.607909,4178.652939,3973.056893,275.401125,2369.253863,541.040216,18.445097,1536.157254,2149.234926,1662.352059


In [16]:
bulk_ind2patient = {i:idx for i, idx in enumerate(data.index)}
data_tensor = torch.from_numpy(data.to_numpy())

In [17]:
data_dict = {bulk_ind2patient[idx]:data_tensor[idx,:] for idx in bulk_ind2patient.keys()}

In [20]:
root = 'raw_bulk_emb'
for dr in dr_dict.keys():
    # print("\n", dr.upper())
    data_dict_copy = deepcopy(data_dict)
    if dr != 'mds':
        # Fit reducer
        dr_dict[dr](data_tensor)
        
        # Apply transformations to each sample individually
        # print("Before", data_dict_copy[bulk_ind2patient[1]].size())
        for pid in data_dict_copy.keys():
            data_dict_copy[pid] = dr_dict[dr].transform(data_dict_copy[pid]).squeeze()
        # print("After", data_dict_copy[bulk_ind2patient[1]].size())
    else:
        # Fit reducer
        # print("Before", data_dict_copy[bulk_ind2patient[1]].size())
        _, data_dict_copy = dr_dict[dr](data_tensor, bulk_ind2patient)
        # print("After", data_dict_copy[bulk_ind2patient[1]].size())
    # write_s3(obj={'data':data_dict_copy}, save_name=root + f"_{dr}",folder="embedding_V1")

60 dimensions hold 99.90% of variance.
Effective compression ratio: 0.12%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_18-36_raw_bulk_emb_pca.pkl
Effective compression ratio: 0.13%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_18-36_raw_bulk_emb_umap.pkl
Effective compression ratio: 0.13%
Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-06-01_18-36_raw_bulk_emb_mds.pkl


### **!!! To execute the remaining cells, you must use at least a 128Go instance !!!**
### **Spatial**
Spatial embeddings are of shape `N_spots x Embedding_size`

In [14]:
from gbmhackathon.utils.visium_functions import normalize_anndata_wrapper

In [15]:
pca_spatial = PCA(n_components=64)

In [16]:
dr_dict_spatial = {'pca':pca_spatial, 'umap':umap, 'mds':mds}

In [17]:
# Loading Raw Data from Mosaic Dataset
data = MosaicDataset.load_visium()

No sample list provided. Loading all samples.
Resolution of the spatial image to load:  lowres
You can change the resolution by setting the resolution parameter using the resolution argument.
Loading Visium data, this can take few minutes...
Copying S3 : s3://pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias/03bb30aa-16ed-4b89-913e-fe009db2aabd/Visium/spaceranger_count/HK_G_001a_vis/ to /tmp/visium_data
Copying S3 : s3://pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias/03bb30aa-16ed-4b89-913e-fe009db2aabd/Visium/spaceranger_count/HK_G_002a_vis/ to /tmp/visium_data
Copying S3 : s3://pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias/03bb30aa-16ed-4b89-913e-fe009db2aabd/Visium/spaceranger_count/HK_G_003a_vis/ to /tmp/visium_data
Copying S3 : s3://pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias/03bb30aa-16ed-4b89-913e-fe009db2aabd/Visium/spaceranger_count/HK_G_004a_vis/ to /tmp/visium_data
Copying S3 : s3://pa-3dqtp2dd4t56b7jvg-bx3

In [18]:
# Nomalization using their function
norm_data = normalize_anndata_wrapper(data)

list_patient, list_annData = list(norm_data.keys()), list(norm_data.values())

data_dict = dict(zip(list_patient, list_annData))

In [19]:
data_emb_spatial = load_s3(f"{S3_FOLDER}/2025-03-23_18-32_spatial_emb_V1.pkl")

In [ ]:
root = 'raw_spatial_emb'
for dr in dr_dict_spatial.keys():
    # print("\n", dr.upper())
    data_dict_copy = {}
    if dr != 'mds':
        for pid in tqdm(list(data_dict.keys()), desc=f"Processing using {dr.upper()}.."):
            data_tensor_slide = torch.from_numpy(data_dict[pid].X.toarray())
            dr_dict_spatial[dr](data_tensor_slide)
            data_dict_copy[pid] = dr_dict_spatial[dr].transform(data_tensor_slide, batch=True)
            # print(data_dict_copy[pid].size())
    else:
        for pid in tqdm(list(data_dict.keys()), desc=f"Processing using {dr.upper()}.."):
            data_tensor_slide = torch.from_numpy(data_dict[pid].X.toarray())
            data_dict_copy[pid], _ = dr_dict[dr](data_tensor_slide, {i:'.' for i in range(data_tensor_slide.size(0))})
            # print(data_dict_copy[pid].size())
    write_s3(obj={'data':data_dict_copy}, save_name=root + f"_{dr}",folder="embedding_V1")

Processing using PCA..:   6%|▌         | 5/86 [01:42<30:04, 22.27s/it]

### **scRNA**

In [53]:
data = MosaicDataset.load_singlecell("pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias")


KeyboardInterrupt



In [ ]:
data = remove_duplicate_var_indices(data, 'scRNA')
print(f"DEBUG: Data after removing duplicate indices shape: {data.shape}")
data = replace_nan_var_indices(data, 'scRNA')
print(f"DEBUG: Data after replacing NaN indices shape: {data.shape}")
data = data.obs

In [ ]:
data.head()